In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00


In [2]:
import os
import random
import numpy as np
import torch
import pandas as pd
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

if os.path.exists("/kaggle/input/dsai-unified-dataset"):
    DATA_PATH = "/kaggle/input/dsai-unified-dataset"
else:
    DATA_PATH = "/kaggle/input/datasets/ds22f1001123/dsai-unified-dataset"

print("Using dataset path:", DATA_PATH)

Using dataset path: /kaggle/input/datasets/ds22f1001123/dsai-unified-dataset


In [4]:
train_images_path = os.path.join(DATA_PATH, 'images/train')
all_images = [os.path.join(train_images_path, f) for f in os.listdir(train_images_path) if f.endswith(('.jpg', '.png'))]
all_images.sort()

seed = 2306
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
subset_3000 = random.sample(all_images, 3000)

In [5]:
with open('/kaggle/working/subset_train.txt', 'w') as f:
    for img in subset_3000:
        f.write(f"{img}\n")

In [6]:
with open("data.yaml", "w") as f:
    f.write(f"""
path: {DATA_PATH}

train: /kaggle/working/subset_train.txt
val: images/val

names:
  0: person
  1: table
  2: potted plant
  3: chair
  4: sofa
  5: lamp
  6: door
  7: cabinet
  8: wardrobe
  9: refrigerator
  10: bed
""")

In [7]:
model = YOLO("yolo11s.pt")
DATA_YAML = "/kaggle/working/data.yaml"

In [8]:
# with open("YAML.yaml", "w") as f:
#     f.write("""
# path: /kaggle/input/datasets/ds22f1001123/dsai-unified-dataset

# train: images/train
# val: images/val

# names:
#   0: person
#   1: table
#   2: potted plant
#   3: chair
#   4: sofa
#   5: lamp
#   6: door
#   7: cabinet
#   8: wardrobe
#   9: refrigerator
#   10: bed
#     """)

In [9]:
base_params = dict(
    data=DATA_YAML,
    epochs=80,
    patience=10,
    imgsz=640,
    batch=32,
    device=0,

    fliplr=0.5,
    degrees=10,
    translate=0.15,
    scale=0.5,
    mosaic=1.0,
    mixup=0.2,

    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,

    cos_lr=True,
    close_mosaic=10,
    warmup_epochs=3,

    seed=2306,
    deterministic=True,
    
    project="image_size",
)

In [10]:
param_grid = [
    {"imgsz": 640},
    {"imgsz": 768},
    {"imgsz": 832},
    {"imgsz": 960},
    {"imgsz": 768, "scale": 0.7},
]

In [11]:
results = []

for i, params in enumerate(param_grid):
    print(f"\n\n========== RUN {i+1} ==========\n")

    model = YOLO("yolo11s.pt")
    
    train_params = base_params.copy()
    train_params.update(params)
    train_params["name"] = f"run_{i+1}"
    train_params["exist_ok"] = True
    
    metrics = model.train(**train_params)
    
    results.append({
        "run": i+1,
        **params,
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
    })




========== RUN 1 ==========

Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [12]:
df = pd.DataFrame(results)
df.to_csv("results_scale_tuning.csv", index=False)
print(df)

   run  imgsz     map50  map50_95  precision    recall  scale
0    1    640  0.537338  0.347108   0.631319  0.507647    NaN
1    2    768  0.549790  0.357472   0.662391  0.505091    NaN
2    3    832  0.551735  0.355906   0.656394  0.513816    NaN
3    4    960  0.569285  0.362872   0.637709  0.530594    NaN
4    5    768  0.566962  0.369940   0.630572  0.548334    0.7


In [13]:
# results = model.train(
#     data=DATA_YAML,
#     epochs=50,
#     imgsz=640,
#     batch=32,          # 🔥 important
#     device=0,

#     degrees=0,
#     translate=0,
#     scale=0,
#     fliplr=0,
#     hsv_h=0,
#     hsv_s=0,
#     hsv_v=0,

#     patience=10
# )

In [14]:
# val_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# metrics = val_model.val(data=DATA_YAML)

# print(f"mAP@50-95: {metrics.box.map:.4f}")
# print(f"mAP@50:    {metrics.box.map50:.4f}")
# print(f"Precision: {metrics.box.mp:.4f}")
# print(f"Recall:    {metrics.box.mr:.4f}")

In [15]:
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg

# results_dir = '/kaggle/working/runs/detect/train/'

# plt.figure(figsize=(10, 10))
# img = mpimg.imread(os.path.join(results_dir, 'confusion_matrix.png'))
# plt.imshow(img)
# plt.axis('off')
# plt.title('Confusion Matrix')
# plt.show()

# plt.figure(figsize=(12, 8))
# img = mpimg.imread(os.path.join(results_dir, 'results.png'))
# plt.imshow(img)
# plt.axis('off')
# plt.title('Training Metrics and Loss Curves')
# plt.show()